# Method Discovery from Literature (Jupyter 版)

研究トピックから論文を集め、知識(抽出結果)を蓄積し、**条件ごとの最良手法**を引用つきで出すシステムの全工程です。上から順に実行します。
各ステージは途中で止めても、もう一度実行すれば続きから再開します(結果は `data/db/app.sqlite` に保存)。

| SPEC の段階 | このノートブック |
|---|---|
| A 検索語の設計と決定(定義 → 候補 → 試験 → 決定 → 人の確認) | 3, 4 |
| B 収集 / C 重複除外 | 5 |
| D 判定(include / exclude / unsure、人の上書き) | 6, 7 |
| E 全文の取得と**全文の読み込み**(PubMed Central + Europe PMC) | 8, 9 |
| F 構造化抽出(原文引用で検証) | 10 |
| G 検索基盤 RAG(キーワード + ベクトル、引用つき質問応答) | 11 |
| H 分類体系(提案 → 確認 → 採用) | 12 |
| I 統合・矛盾・目標チェック | 13, 14 |
| 品質管理(レビューシート、ゴールデンセット) | 15 |
| J レポート(完全版 + Gemma設計のサマリー) | 16 |
| 知識駆動ループ、履歴 | 17, 18 |

- 必要なもの: Ollama が起動していること(`qwen3:8b`, `qwen3:14b`, `bge-m3`。サマリーのデザインに `gemma4:12b`)、インターネット(PubMed / PMC / Europe PMC)
- 長い処理(判定・抽出・索引)は数時間かかります。最初は設定セルの上限を小さくして試せます
- 引用はすべて、保存した本文に実在するか自動で検証されます。全文は **商用利用できるCCライセンス**(CC BY など)のものだけ使います

**別のトピックで使うには**: 設定セルの `RESEARCH_QUESTION` に研究の質問を書いて、上から実行するだけです(既定のトピックはありません)。検索語は、定義 → 候補 → 試験(関連率) → 決定の順に自動で設計され、理由つきで記録されます。既存のトピックは `USE_TOPIC` で続けられます。プロファイルと採用基準はLLMが下書きして `profiles/<ID>.json` に保存します。

### 1. 設定
- 動作: どのトピックを扱うか、検索語の設計の調整、各ステージの上限などをまとめて決める。**トピックは次のどれか1つ**で決める: (a) `RESEARCH_QUESTION` に研究の質問を書く(推奨。検索語を自動で設計する)、(b) `USE_TOPIC` で既存のトピックを選ぶ、(c) `MANUAL_TOPIC` で自分の検索式を直接指定する
- 出力: なし

In [ ]:
# 設定(変更したい値はここだけ)
import os
# --- どのトピックを扱うか(次のどれか1つを設定) ---
RESEARCH_QUESTION = ""     # 例: "What is the best sample preparation for measuring cotinine in saliva by LC-MS/MS?"
USE_TOPIC = None           # 既存のトピックを使う(IDまたは名前)。2章で一覧を確認できる
MANUAL_TOPIC = None        # 上級: 検索式を自分で決める  例 {"name": "My topic", "seeds": ["term A AND term B", "term C"]}
# --- 検索語の設計(3〜4章) ---
DEFINITION_EDITS = {}      # 3章の「定義」を直したいとき。例 {"method_terms": ["lc-ms/ms", "hplc-ms/ms"], "exclude_terms": ["veterinary"]}
REDESIGN = False           # True: 既に設計済みのトピックでも、候補を作り直して評価・決定をやり直す
FORCE_KEEP = []            # 設計の結果に関係なく使う検索式(4章の表の query をそのまま書く)
FORCE_DROP = []            # 設計の結果に関係なく使わない検索式
# --- 基準・プロファイル(任意) ---
CRITERIA_FILE = None       # 自分の採用/除外基準(テキストファイル)。None ならLLMが下書きする(既存トピックは既存の基準を使う)
PROFILE_FILE = None        # 編集済みのプロファイル(JSON)。None ならLLMが自動生成する(既存トピックは既存のものを使う)
# --- 各ステージの上限 ---
SCREEN_LIMIT = int(os.environ.get("SCREEN_LIMIT", 500))       # 1回にLLMで判定する論文数の上限
EXTRACT_LIMIT = int(os.environ.get("EXTRACT_LIMIT", 200))     # 1回に抽出する論文数の上限
# --- 確認・質問 ---
VIEW_PMID = None           # 9章で本文を確認する論文のPMID(None なら全文のある最初の論文)
QUESTIONS = []             # 11章: 知識ベースへの質問。例 ["Which internal standards are used for cotinine in urine?"]
EVAL_CASES = []            # 例: [("saliva in-tube SPME biphenyl column", 39275005)]  期待するPMIDが上位に出るか、検索方式を比べる
RUN_ASK = int(os.environ.get("RUN_ASK", 1))                   # 1 で質問応答を実行(LLMを使う)
EXPAND_QUESTIONS = False   # True: 11章で、知識ベースに根拠が無い質問は、質問から新しい検索式を作って論文を追加収集してから答える
APPLY_TAXONOMY = False     # 12章: True にすると、確認済みの新しい分類をプロファイルに採用する
DESIGN_MODEL = None        # サマリーのデザインに使うモデル(None なら config.yaml の design_model = gemma4:12b)
ROUNDS = 2                 # 知識駆動ループの最大ラウンド数
RUN_LOOP = False           # True にすると 17 章でループ全体を自動実行

### 2. 準備
- 動作: プロジェクトの `app/` を読み込み、DBを開き(無ければ作成)、使うモデル名と、これまでのトピックの一覧を確認する
- 出力: モデル名、既存トピックの一覧

In [ ]:
# コードを更新しても、カーネルを再起動せずに自動で読み込み直す(古いモジュールが残るのを防ぐ)
%load_ext autoreload
%autoreload 2
# プロジェクトを読み込み、DBを開く
import sys, json, html, warnings
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display
warnings.filterwarnings('ignore', message='Consider using IPython.display.IFrame')
assert os.path.exists("config.yaml"), "ノートブックはプロジェクトのルートフォルダで開いてください"
sys.path.insert(0, os.getcwd())
from app import db, stages, fulltext, knowledge, export, loop, rag, taxonomy, design, questions, querydesign
from app import profile as prof
from app.config import load_config
conn = db.init_db()
cfg = load_config()
print("LLM:", cfg["models"]["llm"], "| screening:", cfg["screening_model"], "| extraction:", cfg["extraction_model"],
      "| embedding:", cfg["models"]["embedding"], "| design:", DESIGN_MODEL or cfg["design_model"])
print("existing topics (use USE_TOPIC to continue one of them):")
display(pd.read_sql("SELECT id, name, created_at FROM topics ORDER BY id", conn))

### 3. トピックの定義(ステージA-1)
- 動作: `RESEARCH_QUESTION` から、LLMが検索の**定義**を作る: 主題の語(対象と同義語)・手法の語・比較したい条件・除外したい話題。この定義が検索式の材料になる。表示された定義を読んで、直したいときは `DEFINITION_EDITS` を書いてこのセルを再実行する。`USE_TOPIC` / `MANUAL_TOPIC` のときは定義を作らない
- 出力: トピック名、目的、各語のリスト

In [ ]:
# 質問からトピックの定義を作る(人が確認・編集する)
definition = None
if USE_TOPIC is not None:
    print("using an existing topic (USE_TOPIC):", USE_TOPIC)
elif RESEARCH_QUESTION:
    definition = {**querydesign.define(RESEARCH_QUESTION), **DEFINITION_EDITS}
    print("topic:", definition["topic_name"], "\ngoal:", definition["goal"])
    display(pd.DataFrame({k: [", ".join(definition[k])] for k in ("subject_terms", "method_terms", "condition_terms", "exclude_terms")}).T.rename(columns={0: "terms"}))
elif MANUAL_TOPIC:
    print("using manual seeds (MANUAL_TOPIC):", MANUAL_TOPIC["name"])
else:
    raise SystemExit("Set one of RESEARCH_QUESTION / USE_TOPIC / MANUAL_TOPIC in the settings cell (section 1). Existing topics are listed in section 2.")

### 4. 検索語の設計と決定(ステージA-2)
- 動作: 定義から**検索式の候補**を作り(全語の組・最も狭い組・条件ごと・MeSH・LLMの自由な式)、各候補を**試験**する: PubMedの件数、結果から無作為に取った論文をトピックの基準でLLMが判定した**関連率**、既に選んだ式がカバーしていない論文の割合(**新規性**)。基準(`config.yaml` の `query_design`)を満たす式を、関連率の高い順に**採用**し、採用しない式にも**理由**を残す。採用された式だけが5章で収集される。結果を見て、`FORCE_KEEP` / `FORCE_DROP` で人が上書きできる(トピックのプロファイルと採用基準もここでLLMが下書きする。数分かかる)
- 出力: 検索式の一覧(件数・関連率・新規性・採用/不採用と理由)

In [ ]:
# 検索式を設計・試験・決定し、結果を表示する(設計済みのトピックでは、REDESIGN=True のときだけやり直す)
if USE_TOPIC is not None:
    row = conn.execute("SELECT id FROM topics WHERE id=? OR name=?", (USE_TOPIC if isinstance(USE_TOPIC, int) else -1, str(USE_TOPIC))).fetchone()
    assert row, f"topic {USE_TOPIC!r} not found (see the list in section 2)"
    TOPIC_ID = row["id"]
elif definition:
    TOPIC_ID = querydesign.new_topic(conn, definition, open(CRITERIA_FILE).read() if CRITERIA_FILE else None, PROFILE_FILE)
    n_decided = conn.execute("SELECT COUNT(*) FROM queries WHERE topic_id=? AND status IS NOT NULL", (TOPIC_ID,)).fetchone()[0]
    if REDESIGN or not n_decided:
        querydesign.design(conn, TOPIC_ID)
    else:
        print("queries were already designed for this topic (set REDESIGN=True to redo)")
else:
    TOPIC_ID = stages.setup_topic(conn, MANUAL_TOPIC["name"], MANUAL_TOPIC["seeds"], open(CRITERIA_FILE).read() if CRITERIA_FILE else None, PROFILE_FILE)
if FORCE_KEEP or FORCE_DROP:
    print("manual overrides applied:", querydesign.override(conn, TOPIC_ID, FORCE_KEEP, FORCE_DROP))
P = prof.load(conn, TOPIC_ID)
print("topic id:", TOPIC_ID, "| profile: profiles/%d.json" % TOPIC_ID, "| method classes:", [n for n, _ in P["method_classes"]])
pd.set_option("display.max_colwidth", 110)
pd.DataFrame(querydesign.table(conn, TOPIC_ID))

### 5. 収集と重複除外(ステージB・C)
- 動作: 全検索語でPubMedを検索して論文情報を保存(結果は data/raw にキャッシュされ、再実行しても同じIDになる)。DOI/PMID/題名で重複をまとめる
- 出力: 収集件数、広すぎて除外した検索語、重複の件数

In [ ]:
# 収集して重複を除く
r = stages.collect(conn, TOPIC_ID)
print("collected:", r["total_ids"], "new:", r["added"], "| too broad (skipped):", [t for t, n in r["skipped_too_broad"]])
print("dedup merged:", stages.dedup(conn)["merged"])

### 6. 判定(ステージD)
- 動作: 題名・要旨を基準に、LLMが include / exclude / unsure と理由を付ける(トピックの語が無い論文はルールで除外)。題名に手法語がある論文を優先し、`SCREEN_LIMIT` 件まで
- 出力: ラベル別の件数

In [ ]:
# 未判定の論文を判定する(途中で止めても続きから再開できる)
print(stages.screen(conn, TOPIC_ID, limit=SCREEN_LIMIT))

### 7. 判定の確認と上書き
- 動作: unsure(迷った論文)を理由つきで表示する。人が判断を変えたいときは、下のコメントを外して実行する(reviewer=human になり、再実行しても消えない)
- 出力: unsure の一覧

In [ ]:
# unsure の論文と理由を表示する
q = """SELECT p.pmid, p.year, substr(p.title,1,90) AS title, s.label, s.reviewer, s.reason
       FROM screening s JOIN papers p ON p.id=s.paper_id WHERE s.topic_id=? AND s.label='unsure'"""
display(pd.read_sql(q, conn, params=(TOPIC_ID,)))
# 上書きの例(PMID を指定して include / exclude に変える)
# conn.execute("UPDATE screening SET label='include', reviewer='human', reason='manual' WHERE topic_id=? AND paper_id=(SELECT id FROM papers WHERE pmid='12345678')", (TOPIC_ID,)); conn.commit()

### 8. 全文の取得(ステージE)
- 動作: 採用論文について、まず PubMed Central、取れなければ Europe PMC から全文(JATS XML)を取得する(各情報源は一度だけ試す)。**商用利用できるCCライセンス(CC BY / CC BY-SA / CC0 など)の本文だけ**保存し、他は要旨のみ使う。全文が後から手に入った論文は、次の抽出で自動的に全文を使ってやり直す
- 出力: 取得結果の内訳(例 `pmc:ok`, `europepmc:ok`, `europepmc:not_open_access`)

In [ ]:
# PMC → Europe PMC の順に全文(使えるライセンスのみ)を取得する
print(fulltext.fetch_all(conn, TOPIC_ID))

### 9. 全文の読み込み状況(ステージE)
- 動作: 採用論文ごとに「何を読んだか」を一覧にする(全文の有無・情報源・ライセンス・節数・文字数・抽出で読んだ文字数・全文が無い理由)。`VIEW_PMID` の論文は節ごとの冒頭も表示する
- 出力: 全文カバー率、全文が無い理由の内訳、全文のある論文の表、1本の本文プレビュー

In [ ]:
# 全文の読み込み状況を確認する
cov = pd.DataFrame(fulltext.coverage(conn, TOPIC_ID))
n_full = int(cov["full_text"].sum())
print(f"included papers: {len(cov)} | full text read: {n_full} ({n_full / max(len(cov), 1):.0%}) | abstract only: {len(cov) - n_full}")
print("by source:", cov[cov.full_text].source.value_counts().to_dict())
print("why no full text:", cov[~cov.full_text].why_not.value_counts().head(6).to_dict())
print("share of full text read for extraction: %.0f%%" % (100 * cov.chars_read_for_extraction.sum() / max(cov.chars_total.sum(), 1)), "(rest is searchable in section 11)")
display(cov[cov.full_text].drop(columns=["full_text", "why_not"]).head(15))
view = VIEW_PMID or (cov[cov.full_text].pmid.iloc[0] if n_full else None)
if view:
    print("sections of PMID", view)
    display(pd.DataFrame(fulltext.show(conn, view, chars=250)))

### 10. 知識の抽出と蓄積(ステージF)
- 動作: 採用論文ごとに、トピックのプロファイルに沿った項目(試料・対象・手法・詳細・性能・検証の種類)をLLMが抽出する。各項目には原文の引用が必須で、原文に無い引用は却下して null にする。全文があればその Methods / Results も読む。`EXTRACT_LIMIT` 件まで
- 出力: 抽出ステータス別の件数、蓄積された知識の表(先頭20件)

In [ ]:
# 未抽出の採用論文(と、全文が後から取れた論文)を抽出し、使える知識を表にする
print(knowledge.extract_all(conn, TOPIC_ID, EXTRACT_LIMIT))
items = knowledge.load_knowledge(conn, TOPIC_ID)
df = pd.DataFrame([{"pmid": i["pmid"], "year": i["year"], "condition": ", ".join(i["matrices"]), "method class": i["prep"],
                    "detail A": i["column"], "detail B": i["f"]["internal_standard"]["value"],
                    "performance": i["f"]["lloq"]["value"], "quote": i["f"]["sample_preparation"]["quote"]} for i in items])
print(len(df), "papers with usable knowledge")
df.head(20)

### 11. 検索基盤 RAG と質問応答(ステージG)
- 動作: 採用論文の要旨と**全文の全ての節**を約260語のチャンク(節をまたがない)に分け、キーワード検索(SQLite FTS5)とベクトル検索(FAISS + bge-m3)を作る。検索は両者の順位を合わせる(RRF)ハイブリッド。質問には、検索した抜粋だけを根拠にLLMが答え、**根拠の引用が抜粋に実在するか検証**して付ける。`EVAL_CASES` があれば keyword / vector / hybrid の検索精度を比べる
- 出力: 索引の件数、質問ごとの回答・検証済み根拠・使ったチャンク、検索方式の比較

In [ ]:
# 索引を作り(未作成分のみ)、質問に答える
print(rag.build_index(conn, TOPIC_ID))
for question in QUESTIONS if RUN_ASK else []:
    out = questions.ask_with_expansion(conn, TOPIC_ID, question, expand="auto" if EXPAND_QUESTIONS else False)
    if out["expanded"]:
        print("(no verified evidence in the knowledge base: collected more papers from the question)", out["expanded"])
    res = out["final"]
    print("\nQ:", question, "\nA:", res["answer"])
    for e in res["evidence"]:
        print(f"   [{e['n']}] PMID {e['pmid']} {'verified' if e['verified'] else 'NOT verified'}: {e['quote']}")
    display(pd.DataFrame([{"pmid": c["pmid"], "year": c["year"], "section": c["section"], "via": c["via"], "score": c["score"], "text": c["text"][:110]} for c in res["chunks"]]))
if EVAL_CASES:
    print("hit@5 (keyword / vector / hybrid):", rag.evaluate(conn, TOPIC_ID, EVAL_CASES))

### 12. 分類体系(ステージH)
- 動作: プロファイルの手法クラスから分類体系を作り、論文にタグを付ける(規則ベースで、再実行しても同じ)。クラスに入らず「other」になった論文は、埋め込み + k-means(FAISS)でまとめて、LLMが新しい分類名・正規表現・定義を**提案**する。提案は人が確認し(`accept` を編集)、`APPLY_TAXONOMY = True` で採用するとプロファイルに保存される
- 出力: タグ付けの件数、分類ごとの論文数、新しい分類の提案の表

In [ ]:
# 分類体系を作ってタグ付けし、「other」から新しい分類を提案する
print(taxonomy.tag_papers(conn, TOPIC_ID))
display(pd.read_sql("SELECT t.name, COUNT(g.paper_id) AS papers FROM taxonomy t LEFT JOIN paper_tags g ON g.taxonomy_id=t.id "
                    f"WHERE t.topic_id={TOPIC_ID} GROUP BY t.id ORDER BY papers DESC", conn))
proposals = taxonomy.propose(conn, TOPIC_ID)
display(pd.DataFrame(proposals).drop(columns=["examples"]) if proposals else "no proposals (fewer than 4 papers in 'other')")
# 確認: 不要な提案は accept を False に、名前や正規表現を直したいときは proposals[i]["name"] などを書き換える
# for p in proposals: p["accept"] = p["name"] in ("...",)
if APPLY_TAXONOMY and proposals:
    print(taxonomy.apply(conn, TOPIC_ID, proposals))

### 13. 統合(ステージI)と目標チェック
- 動作: 蓄積した知識を「手法クラス × 条件」に集計し、条件ごとに最良の手法を選ぶ(証拠グレードA〜D、論文数の順。「other」は最後)。同じ条件・同じクラスで性能値が10倍以上ずれる**矛盾する論文**も列挙する。目標(カバー範囲・引用の検証・推奨の完全性など)を自動チェックし、結果を日時つきフォルダに保存する準備をする
- 出力: 目標チェックの表、条件ごとの推奨の表、矛盾の表

In [ ]:
# 統合して目標をチェックし、保存先フォルダを作る
syn = knowledge.synthesize(conn, TOPIC_ID)
goal = knowledge.check_goal(conn, TOPIC_ID, syn)
display(pd.DataFrame([{"check": k, "result": "PASS" if v["ok"] else "FAIL", "detail": v["detail"]} for k, v in goal.items()]))
rec = pd.DataFrame([{"condition": m, "papers": r["n_matrix_total"], "best method class": r["best"], "n": r["n"], "grade": r["grade"],
                     "detail A": r["column"], "detail B": (r["internal_standard"] or "")[:60]}
                    for m, r in sorted(syn["recs"].items(), key=lambda kv: -kv[1]["n_matrix_total"]) if r["n_matrix_total"] >= 2])
display(rec)
conf = [{"condition": m, "unit": c["unit"], "low": c["low"][0], "low PMID": c["low"][1], "high": c["high"][0], "high PMID": c["high"][1], "ratio": c["ratio"]}
        for m, r in syn["recs"].items() if r["n_matrix_total"] >= 2 for c in r["conflicts"]]
print("conflicting reports:")
display(pd.DataFrame(conf) if conf else "none flagged")
folder = export.export_search(conn, TOPIC_ID)
print("snapshot folder:", folder)

### 14. 図: 手法クラス × 条件(プロファイルの定義による)
- 動作: 各組み合わせを扱った論文数をヒートマップにする(地味な単色。軸の名前と分類はトピックのプロファイルによる)
- 出力: ヒートマップ

In [ ]:
# 論文数のヒートマップを描く
classes = [n for n, _ in P["method_classes"]] + ["other"]
mats = sorted({k[0] for k in syn["cells"]})
data = [[syn["cells"].get((m, c), 0) for c in classes] for m in mats]
fig, ax = plt.subplots(figsize=(10, 4.5))
im = ax.imshow(data, cmap="Blues")
ax.set_xticks(range(len(classes)), classes, rotation=40, ha="right"); ax.set_yticks(range(len(mats)), mats)
for i in range(len(mats)):
    for j in range(len(classes)):
        if data[i][j]: ax.text(j, i, data[i][j], ha="center", va="center", fontsize=8)
ax.set_title("Number of papers: %s x %s" % (P["roles"]["matrix"]["label"], P["roles"]["sample_preparation"]["label"])); fig.colorbar(im, label="papers"); plt.tight_layout(); plt.show()

### 15. 品質管理(SPEC §12)
- 動作: (1) 抽出結果の約7%の論文を無作為に選び、人が正誤を書き込む**レビューシート(CSV)**を保存する。`correct (y/n)` 列に y / n を書いて保存し、下のセルを再実行すると**抽出の正解率(目標85%以上)**を計算する。(2) `tests/golden/<トピック名>.csv`(列 `pmid`)があれば、既知の関連論文を収集段階で何%見つけたか(**recall、目標90%以上**)を計算する。無ければ「未測定」と表示する(数値は作らない)
- 出力: レビューシートの場所と、記入済みなら正解率。ゴールデンセットがあれば recall

In [ ]:
# レビューシートを作り(記入済みなら採点し)、ゴールデンセットがあれば recall を測る
sheet = folder / "review_sheet.csv"
print(knowledge.review_sheet(conn, TOPIC_ID, folder))
print("accuracy (S3):", knowledge.score_review_sheet(sheet))     # 記入前は checked=0 / accuracy=None
topic_name = conn.execute("SELECT name FROM topics WHERE id=?", (TOPIC_ID,)).fetchone()["name"]
print("recall (S1):", stages.recall(conn, topic_name, TOPIC_ID))

### 16. レポート(ステージJ)
- 動作: 検証済みのデータから、(1) **完全版レポート** `report.html`(一覧・詳細と原文引用・行列表・分類・矛盾・検索記録・目標チェック・限界・参考文献)と、(2) **サマリーレポート** `summary.html` を保存する。サマリーのデザイン(HTML/CSSの雛形)は **Gemma(`gemma4:12b`)** が設計し、数値・引用などの内容はPythonが検証済みデータで埋める(モデルは内容を作れない)。雛形はスクリプト・外部通信がないか検査してキャッシュする。モデルが未導入なら既定デザインで作り、その旨を表示する
- 出力: 保存先のパス、デザインの取得元(generated / cached / builtin)、サマリーの表示

In [ ]:
# 完全版レポートとサマリーレポートを保存して表示する
knowledge.write_report(conn, TOPIC_ID, folder, syn, goal, "notebook")
s = design.write_summary(conn, TOPIC_ID, folder, syn, goal, "notebook", model=DESIGN_MODEL)
print("full report:", folder / "report.html")
print("summary:", s["path"], "| design:", s["design_source"])
display(HTML('<iframe srcdoc="' + html.escape((folder / "summary.html").read_text(), quote=True) + '" width="100%" height="900"></iframe>'))

### 17. 知識駆動ループ(任意)
- 動作: `RUN_LOOP = True` のとき、知識の穴(論文が少ない条件、頻出する未検索の対象)から新しい検索語を作り、収集 → 判定 → 全文 → 抽出 → 統合 → レポートを `ROUNDS` 回まで繰り返す。新しい検索語が出なくなったら収束して止まる。数時間かかる
- 出力: ラウンドごとの経過と目標チェック

In [ ]:
# 知識に基づいて論文を追加選択しながら、全工程を繰り返す
if RUN_LOOP:
    loop.run_loop(conn, TOPIC_ID, rounds=ROUNDS, extract_limit=EXTRACT_LIMIT, screen_limit=SCREEN_LIMIT)
else:
    print("RUN_LOOP=False: skipped")

### 18. 履歴
- 動作: これまでの検索と保存済みスナップショットを一覧する(過去のレポートはフォルダの `report.html` / `summary.html` で、いつでも開ける。ブラウザで http://localhost:8000/ の履歴ページ(`./run_local.sh serve`)からも見られる)
- 出力: 検索ごとの件数と保存済みスナップショット

In [ ]:
# 履歴を表にする
hist = export.list_searches(conn)
pd.DataFrame([{"id": h["id"], "search": h["name"], "collected": h["counts"]["collected"], "include": h["counts"]["include"],
               "unsure": h["counts"]["unsure"], "exclude": h["counts"]["exclude"], "not screened": h["counts"]["not_screened"],
               "snapshots": len(h["snapshots"]), "latest": h["snapshots"][0] if h["snapshots"] else ""} for h in hist])